# Part 9, notebook 4 of 5: does the human step help, and what does it cost?

Companion to *Human in the Loop*. The five notebooks follow the post in order. Each explains the idea with a daily-life picture, shows the real source, and runs it. **Cells marked "Your turn" are for you to change and rerun.**

| # | Notebook | Post section | Needs the model server? |
|---|---|---|---|
| 09a | `09a_where_a_human_belongs` | Where a person belongs in the pipeline | no |
| 09b | `09b_pause_and_resume` | Pause, ask, resume: `interrupt()` | no |
| 09c | `09c_who_to_ask` | Which cases to send: the review budget | no (reads the saved Part 8 runs) |
| 09d | `09d_does_the_human_help` | Does the human step help, and what does it cost? | no (reads the saved Part 7 runs) |
| 09e | `09e_queue_and_failures` | The review queue and what goes wrong | one cell uses the model server |

**The picture.** A second proofreader. If the second proofreader catches typos you would have shipped, the book gets better. If they skim and initial every page, you pay for a signature and gain nothing. And if they "fix" correct sentences, the book gets worse. The experiment: put a simulated reviewer behind the Part 7 pipeline and measure all three.

**What is simulated.** The reviewer knows the exact answer (the dataset has it) and makes a mistake on each box with probability `error_rate`. That is a model of a person, not a person; the error rate is a parameter you change. The time per image (30 s) is an **assumption**, not a measurement.

In [1]:
import sys, json, os
from pathlib import Path
os.environ.setdefault("AGENTIC_ZERO_MODEL", "qwen3.8-27b")   # the model name the local vLLM server serves (only 09e calls it)
import warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve()))                # this notebook lives in literate/, the code in notebooks/agentic-zero-to-advanced/
OUT = Path("..").resolve() / "out"
%matplotlib inline
import hitl_eval as ev
from hitl_graph import POLICIES
import matplotlib.pyplot as plt, statistics as st
SUMMARY = json.loads((OUT / "hitl_summary.json").read_text())      # written by hitl_summary.py: the numbers the post uses
print("assumed minutes per image:", ev.SECONDS_PER_IMAGE / 60)

assumed minutes per image: 0.5


## 1. Four policies, a perfect reviewer

`P` is precision (of the boxes reported, how many are real), `R` is recall (of all real voids, how many were reported). `ceiling` is the best recall **any** reviewer could reach: the detector never put a box on the rest, and nobody can approve a box that does not exist.

In [2]:
print(f"{'run / policy':24s} {'images shown':>12s} {'minutes':>8s} {'P':>6s} {'R':>6s} {'ceiling R':>10s} {'false alarms left':>18s}")
for k, v in SUMMARY["policies"].items():
    print(f"{k:24s} {v['sent_to_human']:12.0f} {v['human_minutes']:8.1f} {v['precision']:6.3f} {v['recall']:6.3f} {v['ceiling_recall']:10.3f} {v['fp']:18.0f}")

run / policy             images shown  minutes      P      R  ceiling R  false alarms left
band/none                           0      0.0  0.388  0.160      0.209                 98
band/escalate_only                  3      1.5  0.425  0.160      0.209                 84
band/all_nonpass                   96     48.0  1.000  0.186      0.209                  0
band/all_plus_audit               111     55.5  1.000  0.186      0.209                  0
llm/none                            0      0.0  0.411  0.155      0.209                 86
llm/escalate_only                  36     17.8  0.570  0.162      0.209                 47
llm/all_nonpass                    91     45.5  1.000  0.181      0.209                  0
llm/all_plus_audit                105     52.5  1.000  0.181      0.209                  0


What this says:

- A careful person clears **every** false alarm that reaches them: precision goes from about 0.4 to 1.0.
- Recall barely moves (0.155 to 0.181). It is capped at 0.209 because the detector misses most voids. **A reviewer fixes what the pipeline reports; they cannot find what was never boxed.** If recall is the problem, the fix is upstream, not a person.
- `escalate_only` is cheap for the rule (3 images) and half-useful for the language model (36 images, precision 0.41 to 0.57).
- The audit row: 1 in 10 of the passes were looked at (15 images) and the reviewer found **nothing**. That is reassuring but is only 15 images, not a proof that the passes are clean.

## 2. The reviewer's quality is the whole story

Same policy (`all_nonpass`), now the reviewer errs on a fraction of boxes. Two designs:

- `scope=all`: the reviewer judges every candidate of the image (can rescue a rejected box, but also **invents** defects from the many false alarms).
- `scope=kept`: the reviewer judges only the boxes the pipeline kept.

The grey line is the pipeline with **no** person. Anywhere a curve is below it, the reviewer makes the result worse.

In [3]:
errs = sorted({float(k.split("/")[1]) for k in SUMMARY["noise_sweep"]})
none_p = SUMMARY["policies"]["llm/none"]["precision"]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for scope, color in (("all", "#e8590c"), ("kept", "#1971c2")):
    ax[0].plot(errs, [SUMMARY["noise_sweep"][f"{scope}/{e}"]["precision"] for e in errs], "o-", color=color, label=f"scope={scope}")
    ax[1].plot(errs, [SUMMARY["noise_sweep"][f"{scope}/{e}"]["recall"] for e in errs], "o-", color=color, label=f"scope={scope}")
ax[0].axhline(none_p, color="#868e96", ls="--", label="no person at all")
ax[1].axhline(SUMMARY["policies"]["llm/none"]["recall"], color="#868e96", ls="--", label="no person at all")
ax[0].set_title("precision"); ax[1].set_title("recall")
for a in ax: a.set_xlabel("reviewer error rate per box"); a.legend(fontsize=8)
plt.tight_layout(); plt.show()
for e in errs:
    a, k = SUMMARY["noise_sweep"][f"all/{e}"], SUMMARY["noise_sweep"][f"kept/{e}"]
    print(f"error {e:4.2f}  scope=all P {a['precision']:.3f} R {a['recall']:.3f}   scope=kept P {k['precision']:.3f} R {k['recall']:.3f}")

error 0.00  scope=all P 1.000 R 0.181   scope=kept P 1.000 R 0.155
error 0.02  scope=all P 0.897 R 0.179   scope=kept P 0.950 R 0.153
error 0.05  scope=all P 0.764 R 0.172   scope=kept P 0.892 R 0.146
error 0.10  scope=all P 0.647 R 0.165   scope=kept P 0.832 R 0.141
error 0.15  scope=all P 0.541 R 0.156   scope=kept P 0.770 R 0.131
error 0.20  scope=all P 0.453 R 0.144   scope=kept P 0.692 R 0.121
error 0.30  scope=all P 0.328 R 0.127   scope=kept P 0.581 R 0.109
error 0.40  scope=all P 0.242 R 0.108   scope=kept P 0.481 R 0.093


The trade is clear:

- A reviewer who errs on 1 box in 5 (`0.20`) with `scope=all` gives precision 0.45: **no better than having no reviewer** (0.41). The same reviewer with `scope=kept` still gives 0.69.
- `scope=kept` gives up the rescues (recall 0.155 instead of 0.181 for a perfect reviewer) and in exchange it is much harder to hurt the result. Showing a person fewer boxes to judge is a safety feature.

## 3. The rubber stamp

A reviewer who approves whatever the pipeline kept, without looking:

In [4]:
rs = SUMMARY["rubber_stamp"]; none = SUMMARY["policies"]["llm/none"]
print("rubber stamp :", rs)
print("no person    : precision", none["precision"], "recall", none["recall"], "minutes", none["human_minutes"])
print(f"-> the rubber stamp costs {rs['human_minutes']:.0f} minutes and changes nothing.")

rubber stamp : {'precision': 0.411, 'recall': 0.155, 'sent_to_human': 91, 'human_minutes': 45.5}
no person    : precision 0.411 recall 0.155 minutes 0.0
-> the rubber stamp costs 46 minutes and changes nothing.


A queue with a rubber-stamp reviewer looks like human oversight and is not. The guard against it is to **measure the reviewer**: how often they overturn the pipeline (`approve` vs `reject`/`edit`), and to seed a few boxes with known answers. Section 5 of the post says what I would log.

## 4. Is the real graph the same as the simulation?

The numbers above come from `simulate()`, which applies the policy to a *saved* pipeline run. The real thing is the LangGraph with `interrupt()`. This cell runs the **real graph** on the 200 images (rule classifier, offline) and compares it with `simulate()` on the saved rule run, for a reviewer who makes mistakes.

In [5]:
import time
rev = ev.Reviewer(error_rate=0.1, seed=3)
t = time.time()
real = ev.run_graph("all_nonpass", rev)
sim = ev.simulate(ev.load_saved("band"), "all_nonpass", rev)
same = [sorted(a["final"]) for a in real["rows"]] == [sorted(b["final"]) for b in sim["rows"]]
print(f"real graph: {real['summary']['sent_to_human']} images paused, P {real['summary']['precision']} R {real['summary']['recall']}  ({time.time()-t:.1f} s)")
print(f"simulation: {sim['summary']['sent_to_human']} images paused, P {sim['summary']['precision']} R {sim['summary']['recall']}")
print("identical final boxes on all 200 images:", same)

real graph: 96 images paused, P 0.68 R 0.171  (1.4 s)
simulation: 96 images paused, P 0.68 R 0.171
identical final boxes on all 200 images: True


## 5. Your turn

Change the three variables and rerun. `RUN` is `band` (rule) or `llm` (saved language-model run, 3 repeats averaged).

In [6]:
RUN, POLICY, ERROR, SCOPE = "llm", "all_nonpass", 0.10, "kept"      # <- change me

runs = [ev.load_saved(RUN, i) for i in range(1 if RUN == "band" else 3)]
S = [ev.simulate(r, POLICY, ev.Reviewer(error_rate=ERROR, seed=s, scope=SCOPE))["summary"] for r in runs for s in range(5)]
n = lambda k: st.mean(x[k] for x in S)
base = ev.simulate(runs[0], "none", ev.Reviewer())["summary"]
print(f"{RUN} / {POLICY} / error {ERROR} / scope {SCOPE}")
print(f"  images a person sees: {n('sent_to_human'):.0f}  ({n('human_minutes'):.0f} min at {ev.SECONDS_PER_IMAGE} s each)")
print(f"  precision {n('precision'):.3f} (no person: {base['precision']})   recall {n('recall'):.3f} (no person: {base['recall']}, ceiling {base['ceiling_recall']})")

llm / all_nonpass / error 0.1 / scope kept
  images a person sees: 91  (46 min at 30 s each)
  precision 0.832 (no person: 0.411)   recall 0.141 (no person: 0.155, ceiling 0.209)
